# Telco Customer Churn - Random Forest

Predict whether a customer will leave (Churn = 1).

> If you see *No module named ...* or *file not found*, open the project folder itself in Jupyter/VS Code (or run `import os; os.chdir('/path/to/telco-churn-prediction')`).

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from TelcoFeatures import add_features, clean_data, encode
from helper_utils import plot_importance, show_results

## 1. Load the data

Change the file name or path if your CSV is somewhere else.

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

## 2. Clean, add features, encode

In [ ]:
df = clean_data(df)
df = add_features(df)
data = encode(df)

X = data.drop(columns='Churn')
y = data['Churn']
print(X.shape, '| churn rate:', round(y.mean(), 3))

## 3. Train / test split

`stratify=y` keeps the same churn rate in both parts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

## 4. Baseline: Logistic Regression

In [ ]:
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))
baseline.fit(X_train, y_train)
show_results(baseline, X_test, y_test)

## 5. Random Forest (small grid search)

`class_weight='balanced'` gives the rarer class (churners) more weight. The grid search tries every setting below with 5-fold cross-validation.

In [ ]:
grid = {'n_estimators': [200, 400], 'max_depth': [8, 12, None], 'min_samples_leaf': [1, 5]}
search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    grid, cv=5, scoring='roc_auc')
search.fit(X_train, y_train)

print('Best settings:', search.best_params_)
print('CV ROC-AUC:', round(search.best_score_, 3))
rf = search.best_estimator_

## 6. Results on the test set

In [ ]:
show_results(rf, X_test, y_test)

## 7. Most important features

In [ ]:
plot_importance(rf, X.columns)

## 8. Conclusions

Write your own: did the Random Forest beat Logistic Regression? Which features matter most?